In [2]:
!pip install pandas scikit-learn nltk

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 773.1 kB/s eta 0:00:13
   -- ------------------------------------- 0.5/9.9 MB 773.1 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/9.9 MB 796.7 kB/s eta 0:00:12
   ---- ----------------------------------- 1.0/9.9 MB 846.7 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/9.9 MB 888.0 kB/s eta 0:00:10
   ------ --------------------------------- 1.6/9.9 MB 924.6 kB/s eta 0:00:09
   ------- -------------------------------- 1.8/9.9 MB 961.4 kB/s eta 0:00:09
   -------- ------------------------------- 2.1/9.9 MB 994.3 kB/s eta 0:00:08
   --------- ------------------------------ 2.4/9.9 MB 1.0 MB/s eta 0:00:08
   ---------- ----------


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [6]:
import nltk

# Download the stopwords list for filtering common words
nltk.download('stopwords')

# Download the WordNet database used for lemmatization
nltk.download('wordnet')

# Download a helper resource for lemmatization (omw = Open Multilingual Wordnet)
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


True

In [7]:
NLTK_STOPWORDS = set(stopwords.words('english'))
NLTK_LEMMATIZER = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = [NLTK_LEMMATIZER.lemmatize(word)
             for word in text.split()
             if word not in NLTK_STOPWORDS and len(word) > 1]
    return ' '.join(words)

In [8]:
if __name__ == "__main__":
    print("Starting Spam Detection Training...")

    df = pd.read_csv("spam_data.csv", encoding='latin-1')

    df = df[['sms', 'label']]
    df.dropna(subset=['sms', 'label'], inplace=True)
    df = df[df['sms'].apply(lambda x: isinstance(x, str))]
    df['label'] = pd.to_numeric(df['label'], errors='coerce').astype(int)

    df['processed_sms'] = df['sms'].apply(preprocess_text)

    X = df['processed_sms']
    y = df['label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    pipeline = Pipeline([
        ('tfidf_vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1, 1))),
        ('logistic_classifier', LogisticRegression(random_state=42, solver='liblinear', C=1.0, penalty='l2'))
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    print("\nModel Performance:")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
    print(f"Precision: {precision_score(y_test, predictions):.4f}")

Starting Spam Detection Training...

Model Performance:
Accuracy: 0.9699
Precision: 0.9932


E:\spam-detection-project\venv1\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
